# Model Comparison (Member 2)

This notebook compares multiple classification models on the Telco churn dataset to identify the best-performing model. We train Logistic Regression, Random Forest, and Gradient Boosting, then evaluate them on the same test set.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score
)
from imblearn.over_sampling import SMOTE

## 1. Load Preprocessed Data

In [ ]:
# Reconstruct feature names
df = pd.read_csv("../data/telco.csv")
df.drop("customerID", axis=1, inplace=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df = pd.get_dummies(df, drop_first=True)
X = df.drop("Churn", axis=1)
feature_names = X.columns

# Load preprocessed train/test split
with open("../models/processed_data.pkl", "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Original training set: {X_train.shape[0]} samples")
print(f"After SMOTE: {X_train_sm.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 2. Define and Train Models

We compare three models:
- **Logistic Regression**: Linear model, fast, interpretable
- **Random Forest**: Ensemble of decision trees, handles non-linear patterns
- **Gradient Boosting**: Sequential ensemble, often best accuracy

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_sm, y_train_sm)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        "accuracy": round(accuracy_score(y_test, y_pred) * 100, 2),
        "precision": round(precision_score(y_test, y_pred) * 100, 2),
        "recall": round(recall_score(y_test, y_pred) * 100, 2),
        "f1": round(f1_score(y_test, y_pred) * 100, 2),
        "roc_auc": round(roc_auc_score(y_test, y_prob), 4),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
    }
    
    print(f"  Accuracy: {results[name]['accuracy']}% | Precision: {results[name]['precision']}% | Recall: {results[name]['recall']}% | F1: {results[name]['f1']}% | AUC: {results[name]['roc_auc']}")

print("\nAll models trained.")

## 3. Comparison Table

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df.index.name = "Model"
print(comparison_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].to_string())

## 4. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(10, 7))
colors = ["darkorange", "forestgreen", "royalblue"]

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {roc_auc_val:.3f})")

plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random Guess")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../models/roc_comparison.png", dpi=150)
plt.show()

## 5. Metric Comparison Bar Chart

In [ ]:
metrics_to_plot = ["accuracy", "precision", "recall", "f1"]
plot_df = comparison_df[metrics_to_plot].reset_index()
plot_melted = plot_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_melted, x="Metric", y="Score", hue="Model", ax=ax)
ax.set_title("Model Comparison — Classification Metrics (%)")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 100)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../models/model_comparison.png", dpi=150)
plt.show()

## 6. Select Best Model & Save Artifacts

We select the model with the highest **F1 score** as the best model, since it balances precision and recall — both important for churn prediction.

In [ ]:
# Find best model by F1
best_name = max(results, key=lambda k: results[k]["f1"])
best_model = models[best_name]
best_metrics = results[best_name]

print(f"Best Model: {best_name}")
print(f"  Accuracy:  {best_metrics['accuracy']}%")
print(f"  Precision: {best_metrics['precision']}%")
print(f"  Recall:    {best_metrics['recall']}%")
print(f"  F1 Score:  {best_metrics['f1']}%")
print(f"  ROC AUC:   {best_metrics['roc_auc']}")

# Save best model
with open("../models/best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# Save comparison results
with open("../models/comparison.json", "w") as f:
    json.dump(results, f, indent=2)

# Update main model and metrics if best model outperforms current
with open("../models/model.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("../models/feature_names.pkl", "wb") as f:
    pickle.dump(feature_names, f)
with open("../models/metrics.json", "w") as f:
    json.dump(best_metrics, f, indent=2)

print(f"\nSaved: best_model.pkl, comparison.json, model.pkl, metrics.json")